In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

In [16]:
pip install plotly

Note: you may need to restart the kernel to use updated packages.


In [17]:
# Load CSV files
wt = pd.read_csv("/Users/juliemcdonald/Documents/flamholz_lab/RNASeq_testing/wt/wt_final_gene_counts.csv")
lim = pd.read_csv("/Users/juliemcdonald/Documents/flamholz_lab/RNASeq_testing/lim/lim_final_gene_counts.csv")

# Rename count columns before merging
wt = wt.rename(columns={"count": "wt_counts"})
lim = lim.rename(columns={"count": "lim_counts"})

# Merge on shared gene info columns (outer join keeps all genes from both files)
merged = pd.merge(
    wt[["gene_id", "gene_name", "product", "organism", "wt_counts"]],
    lim[["gene_id", "lim_counts"]],
    on="gene_id",
    how="inner"
)

# Reorder columns
merged = merged[["gene_id", "gene_name", "product", "organism", "wt_counts", "lim_counts"]]

# Save to Excel
merged.to_csv("merged_gene_counts.csv", index=False)

print(f"Done! {len(merged)} genes written to merged_gene_counts.csv")

Done! 16505 genes written to merged_gene_counts.csv


In [18]:
import plotly.express as px

# Keep only genes with counts > 0 in both conditions
df = merged[(merged.wt_counts > 0) & (merged.lim_counts > 0)].copy()

# Plot
fig = px.scatter(
    df,
    x="wt_counts",
    y="lim_counts",
    hover_name="gene_name",
    hover_data={"gene_id": True, "product": True, "organism": True,
                "wt_counts": True, "lim_counts": True},
    log_x=True,
    log_y=True,
    opacity=0.6,
    labels={"wt_counts": "WT counts (log10)", "lim_counts": "LIM counts (log10)"},
    title=f"WT vs LIM gene counts — {len(df):,} genes",
)

# 1:1 reference line
axis_min = min(df["wt_counts"].min(), df["lim_counts"].min())
axis_max = max(df["wt_counts"].max(), df["lim_counts"].max())


# Linear fit on log10-transformed values
log_wt = np.log10(df["wt_counts"])
log_lim = np.log10(df["lim_counts"])
coeffs = np.polyfit(log_wt, log_lim, 1)
slope, intercept = coeffs

x_fit = np.linspace(log_wt.min(), log_wt.max(), 200)
y_fit = slope * x_fit + intercept

fig.add_scatter(
    x=10**x_fit,
    y=10**y_fit,
    mode="lines",
    line=dict(color="orange", width=2),
    name=f"Linear fit (slope={slope:.2f}, intercept={intercept:.2f})",
)

fig.update_traces(marker=dict(size=5, color="#378ADD"))
fig.update_layout(
    width=750, height=620,
    plot_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="#eeeeee"),
    yaxis=dict(showgrid=True, gridcolor="#eeeeee"),
)

# Works in Jupyter and also saves a standalone HTML file
fig.show()
fig.write_html("scatter_wt_vs_lim.html")
print("Saved: scatter_wt_vs_lim.html")

Saved: scatter_wt_vs_lim.html
